### Prepare config file

Approved Drugs Data has been downloaded from Drugbank (https://go.drugbank.com/releases/latest#open-data). Now we are going to convert it to csv file to run `DORAnet` for all the possible substructures. 

In [24]:
import csv
import pandas as pd
from pathlib import Path
import re
from rdkit import Chem
from rdkit.Chem import PandasTools
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

### Load the approved Drug molecules

In [15]:
drugBankDrugDataDF = pd.read_csv("drugbank_vocabulary.csv", dtype=str)
print(f"Loaded {len(drugBankDrugDataDF)} molecules")
drugBankDrugDataDF

Loaded 19830 molecules


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key
0,DB00001,BTD00024 | BIOD00024,Lepirudin,138068-37-8,Y43GF64R34,"[Leu1, Thr2]-63-desulfohirudin | Desulfatohiru...",NaN
1,DB00002,BTD00071 | BIOD00071,Cetuximab,205923-56-4,PQX0D8J21J,Cetuximab | Cétuximab | Cetuximabum | Chimeric...,NaN
2,DB00003,BTD00001 | BIOD00001,Dornase alfa,143831-71-4,953A26OA1Y,Deoxyribonuclease (human clone 18-1 protein mo...,NaN
3,DB00004,BTD00084 | BIOD00084,Denileukin diftitox,173146-27-5,25E79B5CTM,DAB(SUB 389)IL2 | Denileukin | Denileukin dift...,NaN
4,DB00005,BTD00052 | BIOD00052,Etanercept,185243-69-0,OP401G7OJC,Etanercept | etanercept-szzs | etanercept-ykro...,NaN
...,...,...,...,...,...,...,...
19825,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N
19826,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N
19827,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N
19828,DB31795,NaN,FRAX716,1432908-40-1,NaN,NaN,NaN


### Read Drugbank structure file data (.sdf) format

In [16]:
structureFilePath = "open_structures.sdf"

# RDKit helper that loads SDF into a DataFrame (keeps an RDKit Mol column)
drugBankDrugStructureDataDF = PandasTools.LoadSDF(
    structureFilePath,
    smilesName="smiles",      # adds SMILES column
    molColName="mol",         # RDKit Mol objects
    includeFingerprints=False,
    embedProps=True           # include SDF properties as columns
)

print(f"Loaded {len(drugBankDrugStructureDataDF)} molecules")
drugBankDrugStructureDataDF

[14:37:46] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 36 ignored
[14:37:46] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[14:37:46] ERROR: Could not sanitize molecule ending on line 127274
[14:37:46] ERROR: Explicit valence for atom # 13 Cl, 5, is greater than permitted
[14:37:46] Explicit valence for atom # 19 O, 3, is greater than permitted
[14:37:46] ERROR: Could not sanitize molecule ending on line 173531
[14:37:46] ERROR: Explicit valence for atom # 19 O, 3, is greater than permitted
[14:37:47] Explicit valence for atom # 15 O, 3, is greater than permitted
[14:37:47] ERROR: Could not sanitize molecule ending on line 264332
[14:37:47] ERROR: Explicit valence for atom # 15 O, 3, is greater than permitted
[14:37:47] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 16 ignored
[14:37:47] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 20 ignored.
[14:37:47] Warning: ambiguous stereo

Loaded 14612 molecules


,DRUGBANK_ID,SECONDARY_ACCESSION_NUMBERS,COMMON_NAME,CAS_NUMBER,UNII,SYNONYMS,ID,smiles,mol
0,DB00006,BTD00076; EXPT03302; BIOD00076; DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin; Bivalirudina; Bivalirudinum,,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,<rdkit.Chem.rdchem.Mol object at 0x7fe815a7a960>
1,DB00014,BTD00113; BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin; Goserelina,[NO NAME],CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,<rdkit.Chem.rdchem.Mol object at 0x7fe815a79540>
2,DB00027,BTD00036; BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D; Gramicidin; Gram...,,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,<rdkit.Chem.rdchem.Mol object at 0x7fe80784b060>
3,DB00035,BTD00112; BTD00061; BIOD00112; BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin; 1-(3-mercap...,,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,<rdkit.Chem.rdchem.Mol object at 0x7fe80784b370>
4,DB00050,BTD00115; APRD00686; BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix; Cetrorelixum,,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,<rdkit.Chem.rdchem.Mol object at 0x7fe80784b840>
...,...,...,...,...,...,...,...,...,...
14614,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline; O,n-dipalmitoylhyd...",,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a260>
14615,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid; (r)-gossypol acetic ...,,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a2d0>
14616,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid; (±)-gossypol aceti...,,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a3b0>
14617,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a420>


In [17]:
# Merge on DrugBank ID first
mergedOnDrugBankID = drugBankDrugDataDF.merge(
    drugBankDrugStructureDataDF[['DRUGBANK_ID', 'smiles', 'mol']],
    left_on='DrugBank ID',
    right_on='DRUGBANK_ID',
    how='left',
    suffixes=('', '_fromID')
)
mergedOnDrugBankID = mergedOnDrugBankID.drop(columns=['DRUGBANK_ID'])

# For rows still missing SMILES, try merging on CAS number
missingMask = mergedOnDrugBankID['smiles'].isna()
missingCount = missingMask.sum()
print(f"Matched by DrugBank ID: {len(mergedOnDrugBankID) - missingCount}")
print(f"Still missing after DrugBank ID match: {missingCount}")

if missingCount > 0:
    # Clean CAS columns for matching
    drugBankDrugDataDF_clean = mergedOnDrugBankID[missingMask].copy()
    drugBankDrugDataDF_clean['CAS_clean'] = drugBankDrugDataDF_clean['CAS'].astype(str).str.strip()

    structureDFClean = drugBankDrugStructureDataDF[['CAS_NUMBER', 'smiles', 'mol']].copy()
    structureDFClean['CAS_clean'] = structureDFClean['CAS_NUMBER'].astype(str).str.strip()
    structureDFClean = structureDFClean[
        (structureDFClean['CAS_clean'] != '') &
        (structureDFClean['CAS_clean'] != 'nan')
    ].drop_duplicates(subset='CAS_clean')

    casMerged = drugBankDrugDataDF_clean.merge(
        structureDFClean[['CAS_clean', 'smiles', 'mol']],
        on='CAS_clean',
        how='left',
        suffixes=('', '_fromCAS')
    )

    # Fill missing smiles and mol from CAS match
    mergedOnDrugBankID.loc[missingMask, 'smiles'] = casMerged['smiles_fromCAS'].values
    mergedOnDrugBankID.loc[missingMask, 'mol'] = casMerged['mol_fromCAS'].values

    casMatchCount = casMerged['smiles_fromCAS'].notna().sum()
    print(f"Matched by CAS number: {casMatchCount}")

# Drop helper columns if present
dropCols = [c for c in mergedOnDrugBankID.columns if c.endswith('_clean')]
mergedOnDrugBankID = mergedOnDrugBankID.drop(columns=dropCols, errors='ignore')

drugBankDrugDataDF = mergedOnDrugBankID.copy()

# Summary
totalDrugs = len(drugBankDrugDataDF)
foundCount = drugBankDrugDataDF['smiles'].notna().sum()
missingCount = drugBankDrugDataDF['smiles'].isna().sum()

print(f"\nFinal Summary:")
print(f"  Total drugs: {totalDrugs}")
print(f"  With SMILES: {foundCount} ({foundCount/totalDrugs*100:.2f}%)")
print(f"  Missing SMILES: {missingCount} ({missingCount/totalDrugs*100:.2f}%)")

drugBankDrugDataDF

Matched by DrugBank ID: 14612
Still missing after DrugBank ID match: 5218
Matched by CAS number: 6

Final Summary:
  Total drugs: 19830
  With SMILES: 14618 (73.72%)
  Missing SMILES: 5212 (26.28%)


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,smiles,mol
0,DB00001,BTD00024 | BIOD00024,Lepirudin,138068-37-8,Y43GF64R34,"[Leu1, Thr2]-63-desulfohirudin | Desulfatohiru...",NaN,NaN,NaN
1,DB00002,BTD00071 | BIOD00071,Cetuximab,205923-56-4,PQX0D8J21J,Cetuximab | Cétuximab | Cetuximabum | Chimeric...,NaN,NaN,NaN
2,DB00003,BTD00001 | BIOD00001,Dornase alfa,143831-71-4,953A26OA1Y,Deoxyribonuclease (human clone 18-1 protein mo...,NaN,NaN,NaN
3,DB00004,BTD00084 | BIOD00084,Denileukin diftitox,173146-27-5,25E79B5CTM,DAB(SUB 389)IL2 | Denileukin | Denileukin dift...,NaN,NaN,NaN
4,DB00005,BTD00052 | BIOD00052,Etanercept,185243-69-0,OP401G7OJC,Etanercept | etanercept-szzs | etanercept-ykro...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
19825,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a2d0>
19826,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a3b0>
19827,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,<rdkit.Chem.rdchem.Mol object at 0x7fe80749a420>
19828,DB31795,NaN,FRAX716,1432908-40-1,NaN,NaN,NaN,NaN,NaN


In [20]:
drugBankDrugDataDF_wSMILES.to_csv('drugBankDrugDataDF_wSMILES.csv', index=False, encoding="utf-8")

In [21]:
drugBankDrugDataDF_wSMILES

,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES
0,DB00006,BTD00076 | EXPT03302 | BIOD00076 | DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin | Bivalirudina | Bivalirudinum,OIRCOABEOLEUMC-GEJPAHFPSA-N,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...
1,DB00014,BTD00113 | BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin | Goserelina,BLCLNMBMMGCOAS-URPVMXJPSA-N,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2,DB00027,BTD00036 | BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D | Gramicidin | Gr...,NDAYQJDHGXTBJL-MWWSRJDJSA-N,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3,DB00035,BTD00112 | BTD00061 | BIOD00112 | BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin | 1-(3-merca...,NFLWUMRGJYTJIN-PNIOQBSNSA-N,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...
4,DB00050,BTD00115 | APRD00686 | BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix | Cetrorelixum,SBNPWPIBESPSIF-MHWMIDJBSA-N,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...
...,...,...,...,...,...,...,...,...
14613,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline | O,n-dipalmitoylhy...",QZLXCFQVOCEKSX-NOCHOARKSA-N,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...
14614,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...
14615,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...
14616,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...


### This code searches for SMILES using 8 different strategies across 4 public databases in a cascading order:

- Databases used: PubChem, ChEMBL, NCI Chemical Identifier Resolver (CIR), UniChem (EBI).
- Identifiers tried: InChI Key, drug name, CAS number, UNII code

In [ ]:
print(f"Loaded {len(drugBankDrugDataDF_woSMILES)} drugs from DrugBank")


# =====================================================================
# Session with connection pooling for faster requests
# =====================================================================
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(
    pool_connections=20,
    pool_maxsize=20,
    max_retries=2
)
session.mount('https://', adapter)
session.mount('http://', adapter)


# =====================================================================
# API Functions using session for connection reuse
# =====================================================================

def getSmilesFromPubChemByName(drugName: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{requests.utils.quote(drugName)}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromPubChemByInChIKey(inchiKey: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchiKey}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromPubChemByCAS(cas: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromPubChemByUNII(unii: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{unii}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromCIRByName(drugName: str) -> str | None:
    try:
        url = f"https://cactus.nci.nih.gov/chemical/structure/{requests.utils.quote(drugName)}/smiles"
        response = session.get(url, timeout=10)
        if response.status_code == 200 and len(response.text) < 1000 and '\n' not in response.text.strip():
            return response.text.strip()
    except Exception:
        pass
    return None


def getSmilesFromCIRByCAS(cas: str) -> str | None:
    try:
        url = f"https://cactus.nci.nih.gov/chemical/structure/{cas}/smiles"
        response = session.get(url, timeout=10)
        if response.status_code == 200 and len(response.text) < 1000 and '\n' not in response.text.strip():
            return response.text.strip()
    except Exception:
        pass
    return None


def getSmilesFromUniChemByInChIKey(inchiKey: str) -> str | None:
    try:
        url = f"https://www.ebi.ac.uk/unichem/rest/inchikey/{inchiKey}"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                srcCompoundId = data[0].get('src_compound_id')
                if srcCompoundId:
                    return getSmilesFromPubChemByName(srcCompoundId)
    except Exception:
        pass
    return None


def getSmilesFromChEMBLByName(drugName: str) -> str | None:
    try:
        url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search?q={requests.utils.quote(drugName)}&format=json"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            molecules = data.get('molecules', [])
            if molecules and molecules[0].get('molecule_structures'):
                return molecules[0]['molecule_structures'].get('canonical_smiles')
    except Exception:
        pass
    return None


def canonicalize(smiles: str) -> str | None:
    if smiles is None:
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


def fetchSmiles(row) -> tuple[str | None, str]:
    drugName = str(row.get('Common name', '')).strip()
    inchiKey = str(row.get('Standard InChI Key', '')).strip()
    cas = str(row.get('CAS', '')).strip()
    unii = str(row.get('UNII', '')).strip()

    if drugName.lower() == 'nan' or drugName == '':
        drugName = None
    if inchiKey.lower() == 'nan' or inchiKey == '':
        inchiKey = None
    if cas.lower() == 'nan' or cas == '':
        cas = None
    if unii.lower() == 'nan' or unii == '':
        unii = None

    # 1. PubChem by InChI Key
    if inchiKey:
        smiles = getSmilesFromPubChemByInChIKey(inchiKey)
        if smiles:
            return canonicalize(smiles), "PubChem_InChIKey"

    # 2. PubChem by drug name
    if drugName:
        smiles = getSmilesFromPubChemByName(drugName)
        if smiles:
            return canonicalize(smiles), "PubChem_Name"

    # 3. PubChem by CAS
    if cas:
        smiles = getSmilesFromPubChemByCAS(cas)
        if smiles:
            return canonicalize(smiles), "PubChem_CAS"

    # 4. PubChem by UNII
    if unii:
        smiles = getSmilesFromPubChemByUNII(unii)
        if smiles:
            return canonicalize(smiles), "PubChem_UNII"

    # 5. ChEMBL by drug name
    if drugName:
        smiles = getSmilesFromChEMBLByName(drugName)
        if smiles:
            return canonicalize(smiles), "ChEMBL_Name"

    # 6. CIR by drug name
    if drugName:
        smiles = getSmilesFromCIRByName(drugName)
        if smiles:
            return canonicalize(smiles), "CIR_Name"

    # 7. CIR by CAS
    if cas:
        smiles = getSmilesFromCIRByCAS(cas)
        if smiles:
            return canonicalize(smiles), "CIR_CAS"

    # 8. UniChem by InChI Key
    if inchiKey:
        smiles = getSmilesFromUniChemByInChIKey(inchiKey)
        if smiles:
            return canonicalize(smiles), "UniChem_InChIKey"

    return None, "NotFound"


def fetchSmilesWorker(args):
    """Wrapper for thread pool."""
    idx, row = args
    smiles, source = fetchSmiles(row)
    drugName = str(row.get('Common name', 'Unknown')).strip()
    return idx, smiles, source, drugName


# =====================================================================
# Fetch SMILES using thread pool for parallel API calls
# =====================================================================

totalDrugs = len(drugBankDrugDataDF_woSMILES)
numWorkers = 16  # Number of parallel threads

print(f"\nFetching SMILES for {totalDrugs} drugs using {numWorkers} parallel threads...")
print(f"APIs: PubChem, ChEMBL, CIR, UniChem")
print(f"Identifiers: InChI Key, Drug Name, CAS, UNII")
print("-" * 70)

# Prepare tasks
tasks = [(idx, row) for idx, row in drugBankDrugDataDF_woSMILES.iterrows()]

# Initialize result storage
smilesResults = [None] * totalDrugs
sourceResults = ['NotFound'] * totalDrugs
failedDrugs = []
completedCount = 0

# Run with thread pool
with ThreadPoolExecutor(max_workers=numWorkers) as executor:
    futures = {executor.submit(fetchSmilesWorker, task): task[0] for task in tasks}

    for future in as_completed(futures):
        idx, smiles, source, drugName = future.result()
        smilesResults[idx] = smiles
        sourceResults[idx] = source

        if smiles is None:
            failedDrugs.append(drugName)

        completedCount += 1
        if completedCount % 100 == 0:
            successCount = sum(1 for s in smilesResults if s is not None)
            print(f"  Processed {completedCount}/{totalDrugs} | "
                  f"Found: {successCount} | "
                  f"Missing: {len(failedDrugs)}")

# Add columns
drugBankDrugDataDF_woSMILES['Canonical_SMILES'] = smilesResults
drugBankDrugDataDF_woSMILES['SMILES_Source'] = sourceResults

# =====================================================================
# Summary
# =====================================================================

foundCount = drugBankDrugDataDF_woSMILES['Canonical_SMILES'].notna().sum()
missingCount = drugBankDrugDataDF_woSMILES['Canonical_SMILES'].isna().sum()
foundPercent = (foundCount / totalDrugs) * 100
missingPercent = (missingCount / totalDrugs) * 100

print(f"\n{'=' * 70}")
print(f"SMILES Retrieval Summary:")
print(f"{'=' * 70}")
print(f"  Found:   {foundCount} ({foundPercent:.2f}%)")
print(f"  Missing: {missingCount} ({missingPercent:.2f}%)")

print(f"\nSource breakdown:")
sourceCounts = drugBankDrugDataDF_woSMILES['SMILES_Source'].value_counts()
for source, count in sourceCounts.items():
    print(f"  {source}: {count} ({count/totalDrugs*100:.2f}%)")

if failedDrugs:
    print(f"\nFailed drugs (first 20):")
    for drug in failedDrugs[:20]:
        print(f"  - {drug}")
    if len(failedDrugs) > 20:
        print(f"  ... and {len(failedDrugs) - 20} more")

drugBankDrugDataDF_woSMILES

Loaded 5212 drugs from DrugBank

Fetching SMILES for 5212 drugs using 16 parallel threads...
APIs: PubChem, ChEMBL, CIR, UniChem
Identifiers: InChI Key, Drug Name, CAS, UNII
----------------------------------------------------------------------


### For a broader search space take `targetSmiles` from a data frame

In [23]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from collections import Counter
import pandas as pd
import numpy as np

# Read the CSV file
startersDF = pd.read_csv("drugBankDrugDataDF_wSMILES.csv")
print(f"Loaded {len(startersDF)} molecules")

# Count atoms for each molecule
atomCounts = []

for smi in startersDF['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomCounts.append({'C': 0, 'N': 0, 'O': 0, 'S': 0})
        continue
    
    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    atomCounts.append({
        'C': atomCounter.get('C', 0),
        'N': atomCounter.get('N', 0),
        'O': atomCounter.get('O', 0),
        'S': atomCounter.get('S', 0)
    })

atomCountsDf = pd.DataFrame(atomCounts)
startersDF = pd.concat([startersDF, atomCountsDf], axis=1)

# Print atom count ranges
print(f"\nAtom count ranges across {len(startersDF)} molecules:")
print(f"  C (Carbon):   min = {startersDF['C'].min()}, max = {startersDF['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF['N'].min()}, max = {startersDF['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF['O'].min()}, max = {startersDF['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF['S'].min()}, max = {startersDF['S'].max()}")

# Print suggested max_atoms config (max values + 50% increase)
maxC = startersDF['C'].max()
maxN = startersDF['N'].max()
maxO = startersDF['O'].max()
maxS = startersDF['S'].max()

print(f"\nSuggested max_atoms config (max values + 50% increase for expanded search space):")
print(f"  C: {int(np.ceil(maxC * 1.5))} # Carbon")
print(f"  N: {int(np.ceil(maxN * 1.5))} # Nitrogen")
print(f"  O: {int(np.ceil(maxO * 1.5))} # Oxygen")
print(f"  S: {int(np.ceil(maxS * 1.5))} # Sulfur")

startersDF

Loaded 14618 molecules


[14:43:50] Unusual charge on atom 42 number of radical electrons set to zero
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors



Atom count ranges across 14618 molecules:
  C (Carbon):   min = 0, max = 208
  N (Nitrogen): min = 0, max = 68
  O (Oxygen):   min = 0, max = 110
  S (Sulfur):   min = 0, max = 18

Suggested max_atoms config (max values + 50% increase for expanded search space):
  C: 312 # Carbon
  N: 102 # Nitrogen
  O: 165 # Oxygen
  S: 27 # Sulfur


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES,C,N,O,S
0,DB00006,BTD00076 | EXPT03302 | BIOD00076 | DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin | Bivalirudina | Bivalirudinum,OIRCOABEOLEUMC-GEJPAHFPSA-N,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,98,24,33,0
1,DB00014,BTD00113 | BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin | Goserelina,BLCLNMBMMGCOAS-URPVMXJPSA-N,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,59,18,14,0
2,DB00027,BTD00036 | BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D | Gramicidin | Gr...,NDAYQJDHGXTBJL-MWWSRJDJSA-N,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,96,19,16,0
3,DB00035,BTD00112 | BTD00061 | BIOD00112 | BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin | 1-(3-merca...,NFLWUMRGJYTJIN-PNIOQBSNSA-N,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,46,14,12,2
4,DB00050,BTD00115 | APRD00686 | BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix | Cetrorelixum,SBNPWPIBESPSIF-MHWMIDJBSA-N,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,70,17,14,0
...,...,...,...,...,...,...,...,...,...,...,...,...
14613,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline | O,n-dipalmitoylhy...",QZLXCFQVOCEKSX-NOCHOARKSA-N,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...,37,1,5,0
14614,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,32,0,10,0
14615,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,32,0,10,0
14616,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,29,7,1,1
